In [1]:
import os, sys

PROJECT_ROOT = '/scratch/jq2uw/derm_vlms'
MEDGEMMA_DIR = os.path.join(PROJECT_ROOT, 'medgemma')

if MEDGEMMA_DIR not in sys.path:
    sys.path.insert(0, MEDGEMMA_DIR)

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'

import torch
torch.cuda.empty_cache()

# HF token for gated MedGemma model
sys.path.insert(0, PROJECT_ROOT)
from tokens import HF_TOKEN

from utils import load_model, predict_image, parse_label

print('Loading model...')
model, processor = load_model(hf_token=HF_TOKEN)
print('Model loaded.')


/home/jq2uw/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading model...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|███████████████████████████████████████████████████| 2/2 [00:03<00:00,  1.60s/it]
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Total params: 4,300,079,472
Model loaded.


In [2]:
import pandas as pd
from pathlib import Path

sys.path.insert(0, PROJECT_ROOT)
from data_utils import sample_lesions

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')
IMAGES_DIR = os.path.join(RESULTS_DIR, 'images')

df = pd.read_parquet(os.path.join(PROJECT_ROOT, 'data_share', 'midas_share.parquet'))
print(f'Loaded {len(df)} rows')
print(f'y3 distribution:\n{df["y3"].value_counts()}')

SEED = 42
N_PER_CLASS = 5
df_sample = sample_lesions(df, data_dir=DATA_DIR, output_dir=IMAGES_DIR, n_per_class=N_PER_CLASS, seed=SEED)
df_sample.head()


Loaded 3357 rows
y3 distribution:
y3
malignant    1391
benign       1322
other         644
Name: count, dtype: int64
Sampled 10 lesions (5 per class) -> 30 rows
Classes: {'benign': 15, 'malignant': 15}
Images saved to: /scratch/jq2uw/derm_vlms/results/images


,id,ground_truth,image_mode,image_path,original_image_name,lesion_id
0,1_photo,benign,photo,/scratch/jq2uw/derm_vlms/results/images/1_phot...,s-prd-667118134.jpg,534_left lower leg_no
1,1_dscope,benign,dscope,/scratch/jq2uw/derm_vlms/results/images/1_dsco...,s-prd-667118139.jpg,534_left lower leg_no
2,1_combined,benign,combined,/scratch/jq2uw/derm_vlms/results/images/1_comb...,s-prd-667118134.jpg; s-prd-667118139.jpg,534_left lower leg_no
3,2_photo,benign,photo,/scratch/jq2uw/derm_vlms/results/images/2_phot...,s-prd-505211076.jpg,215_central chest _no
4,2_dscope,benign,dscope,/scratch/jq2uw/derm_vlms/results/images/2_dsco...,s-prd-505211307.jpg,215_central chest _no


In [ ]:
from PIL import Image
from tqdm import tqdm

q_describe = "You are an expert dermatologist. Please describe using dermatological terms to describe the appearance of this lesion."
q_classify = "Please give the top 3 diagnoses in your differential."
q_describe_classify = "You are an expert dermatologist. Please describe using dermatological terms to describe the appearance of this lesion. Please give the top 3 diagnoses in your differential."
results = []

for _, row in tqdm(df_sample.iterrows(), total=len(df_sample)):
    try:
        image = Image.open(row['image_path']).convert('RGB')
    except Exception as e:
        print(f'[SKIP] {row["id"]}: {e}')
        continue

    results.append({
        'id': row['id'],
        'ground_truth': row['ground_truth'],
        'image_mode': row['image_mode'],
        'describe': predict_image(model, processor, image, prompt=q_describe),
        'classify': predict_image(model, processor, image, prompt=q_classify),
        'describe_then_classify': predict_image(model, processor, image, prompt=q_describe_classify),
        'original_image_name': row['original_image_name'],
        'lesion_id': row['lesion_id'],
    })

print(f'Collected {len(results)} predictions')


  0%|                                                                                     | 0/30 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
  3%|██▌                                                                          | 1/30 [00:51<24:59, 51.72s/it]Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


In [ ]:
results_df = pd.DataFrame(results)
col_order = ['id', 'ground_truth', 'image_mode', 'describe', 'classify',
             'describe_then_classify', 'original_image_name', 'lesion_id']
results_df = results_df[col_order]

out_path = os.path.join(RESULTS_DIR, 'medgemma_predictions_paired.csv')
results_df.to_csv(out_path, index=False)
print(f'Saved {len(results_df)} rows to {out_path}')

results_df
